# Bayesian Optimisation: Protrusion Stimulation Parameters

Uses the `BOptGPAX` inter-phase agent to find the optimal optogenetic
stimulation parameters that maximise cell protrusions (measured as area
increase).

**Steerable parameters (BO optimises):**
- `stim_angle` -- angular position of the stimulation patch around the cell
- `stim_radial` -- radial position of the patch from cell centre to edge

**Covariate (observed, not controlled):**
- `baseline_area` -- median cell area during baseline frames

**Objective:** maximise `area_change` (fractional area increase after stimulation)

In [ ]:
import os
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Suppress JAX/XLA debug messages
os.environ["JAX_LOG_COMPILES"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import jax

jax.config.update("jax_log_compiles", False)
for _name in list(logging.Logger.manager.loggerDict):
    if "jax" in _name or "absl" in _name:
        logging.getLogger(_name).setLevel(logging.WARNING)

from rtm_pymmcore.core.data_structures import (
    Channel,
    RTMSequence,
    SegmentationMethod,
)
from rtm_pymmcore.core.controller import Controller
from rtm_pymmcore.core.pipeline import ImageProcessingPipeline
from rtm_pymmcore.agents.bo_optimization import (
    BOptGPAX,
    BO_Parameter,
    BO_Objective,
    BO_Covariate,
)

## Microscope & pipeline setup

Adapt the imports below to your microscope and segmentation engine.
The example below uses the virtual-microscope optogenetic backend;
replace with your real microscope class for actual experiments.

In [ ]:
# --- Microscope ---
# Option A: Simulated (requires `pip install virtual-microscope`)
from virtual_microscope.backends.optogenetic import setup_optogenetic
from rtm_pymmcore.microscope.simulation import UniMMCoreSimulation

core, sim = setup_optogenetic()
mic = UniMMCoreSimulation(mmc=core)

# Option B: Real microscope (uncomment and adapt)
# from rtm_pymmcore.microscope.pymmcore import PyMMCoreMicroscope
# mic = PyMMCoreMicroscope(config_file="path/to/config.cfg")

# --- Segmentation ---
from rtm_pymmcore.segmentation.base import OtsuSegmentator

# For real experiments, use e.g.:
# from rtm_pymmcore.segmentation.cellpose import SegmentorCellpose

# --- Feature extraction & tracking ---
from rtm_pymmcore.feature_extraction.simple import SimpleFE
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy

In [ ]:
# --- Experiment parameters ---
TIME_BETWEEN_TIMESTEPS = 15  # seconds between frames
N_FRAMES_BASELINE = 12  # 3 min baseline
FIRST_FRAME_STIM = N_FRAMES_BASELINE
LAST_FRAME_STIM = 52  # 10 min stimulation (40 frames)
N_FRAMES = 60  # baseline + stim + post-stim observation

# --- Storage ---
path = os.path.join(os.getcwd(), "output_bo_protrusions")

# --- Channels (use Channel dataclass for stim_channels) ---
img_channels = (Channel(config="FITC", exposure=300),)
stim_channel = Channel(config="CyanStim", exposure=500)

# --- Pipeline ---
pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=[
        SegmentationMethod(
            "labels", OtsuSegmentator(), use_channel=0, save_tracked=True
        )
    ],
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(search_range=50),
    # stimulator=StimTwoAxisPercentage(),  # your stim engine
)

## Define the BO agent subclass

Subclass `BOptGPAX` and implement two template methods:
- `_create_events_for_cycle(parameters)` -- build `RTMEvent` objects for one BO iteration
- `_preprocess_results(fov_tracks)` -- extract the objective metric from tracking parquets

In [ ]:
class ProtrusionBO(BOptGPAX):
    """BO agent for protrusion stimulation optimisation.

    Each BO iteration runs a full experiment phase:
    baseline -> stimulation -> observation.
    """

    def __init__(
        self,
        *,
        fovs,
        n_frames,
        first_frame_stim,
        last_frame_stim,
        time_between_timesteps,
        img_channels,
        stim_channel,
        stage_positions,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self._fovs = fovs
        self.n_frames = n_frames
        self.first_frame_stim = first_frame_stim
        self.last_frame_stim = last_frame_stim
        self.time_between_timesteps = time_between_timesteps
        self.img_channels = img_channels
        self.stim_channel = stim_channel
        self.stage_positions = stage_positions
        self._phase_counter = 0

    def _create_events_for_cycle(self, parameters: dict) -> list:
        """Build RTMEvents for one BO iteration."""
        phase_id = self._phase_counter

        acq = RTMSequence(
            time_plan={"interval": self.time_between_timesteps, "loops": self.n_frames},
            stage_positions=self.stage_positions,
            channels=self.img_channels,
            stim_channels=(self.stim_channel,),
            stim_frames=range(self.first_frame_stim, self.last_frame_stim),
            rtm_metadata={
                "phase_name": f"BO_iter_{phase_id}",
                "phase_id": phase_id,
                "stim_angle": parameters["stim_angle"],
                "stim_radial": parameters["stim_radial"],
            },
        )
        return list(acq)

    def _preprocess_results(self, fov_tracks: dict) -> pd.DataFrame:
        """Extract area_change from tracking data."""
        phase_id = self._phase_counter
        self._phase_counter += 1

        results = []
        for fov_idx, df_tracks in fov_tracks.items():
            if df_tracks.empty or "particle" not in df_tracks.columns:
                continue

            for particle_id, grp in df_tracks.groupby("particle"):
                grp = grp.sort_values("timestep")
                if grp["timestep"].nunique() < self.n_frames:
                    continue

                baseline = grp[grp["timestep"] < self.first_frame_stim]
                if baseline.empty:
                    continue
                baseline_area = baseline["area"].median()
                if baseline_area < 50:
                    continue

                response = grp[grp["timestep"] >= self.first_frame_stim]
                if len(response) < 2:
                    continue

                robust_high = (
                    response.sort_values("timestep")["area"]
                    .rolling(2, min_periods=2)
                    .mean()
                    .max()
                )
                area_change = (robust_high - baseline_area) / baseline_area

                stim_angle = (
                    grp["stim_angle"].iloc[0] if "stim_angle" in grp.columns else 0
                )
                stim_radial = (
                    grp["stim_radial"].iloc[0] if "stim_radial" in grp.columns else 0
                )

                results.append(
                    {
                        "stim_angle": stim_angle,
                        "stim_radial": stim_radial,
                        "baseline_area": baseline_area,
                        "area_change": area_change,
                    }
                )

        if not results:
            print(f"Warning: no valid cells in phase {phase_id}")
            return pd.DataFrame()

        df = pd.DataFrame(results)
        print(
            f"  Phase {phase_id}: {len(df)} cells, mean area_change={df['area_change'].mean():.4f}"
        )
        return df

## Configure and run Bayesian Optimisation

In [ ]:
# --- FOV positions (set via napari-micromanager or manually) ---
stage_positions = [(0.0, 0.0, 0.0)]  # replace with your FOV positions

# --- BO configuration ---
bo_params = [
    BO_Parameter(name="stim_angle", bounds=(0.0, 1.0), spacing=0.1),
    BO_Parameter(name="stim_radial", bounds=(0.0, 1.0), spacing=0.1),
]
bo_covariates = [BO_Covariate(name="baseline_area")]
bo_objective = BO_Objective(name="area_change", goal="maximize")

# --- Create agent ---
agent = ProtrusionBO(
    storage_path=path,
    fovs=list(range(len(stage_positions))),
    n_frames=N_FRAMES,
    first_frame_stim=FIRST_FRAME_STIM,
    last_frame_stim=LAST_FRAME_STIM,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    img_channels=img_channels,
    stim_channel=stim_channel,
    stage_positions=stage_positions,
    parameters_to_optimize=bo_params,
    objective_metric=bo_objective,
    bo_covariates=bo_covariates,
    n_iterations=15,
    acquisition_function="ei",
    n_cov_samples=30,
    ei_xi=0.2,
    ei_xi_final=0.01,
    ei_xi_decay_fraction=0.7,
    verbose=True,
)
agent.add_fovs(list(range(len(stage_positions))))

# --- Create controller and register agent ---
ctrl = Controller(mic, pipeline, agent=agent)

# --- Run the BO loop ---
# The agent's run() method calls ctrl.run_experiment() / continue_experiment()
# internally for each BO iteration.
agent.run()

## Visualise results

In [ ]:
if agent.x is not None and agent.y is not None:
    x_data = np.array(agent.x)
    y_data = np.array(agent.y).ravel()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sc = axes[0].scatter(
        x_data[:, 0],
        x_data[:, 1],
        c=y_data,
        cmap="viridis",
        s=40,
        edgecolors="k",
        linewidths=0.5,
    )
    axes[0].set_xlabel("stim_angle")
    axes[0].set_ylabel("stim_radial")
    axes[0].set_title("Measured area_change")
    fig.colorbar(sc, ax=axes[0], label="area_change")

    cumulative_best = np.maximum.accumulate(y_data)
    axes[1].plot(cumulative_best, "o-")
    axes[1].set_xlabel("Measurement index")
    axes[1].set_ylabel("Best area_change so far")
    axes[1].set_title("BO convergence")

    plt.tight_layout()
    plt.show()
else:
    print("No data collected yet.")